In [2]:
! pip install prophet


    extract-msg (<=0.29.*)
                 ~~~~~~~^

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [21]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error

STATE_SUM_COLS = ["avail_bikes", "avail_docks", "tot_docks"]  # only if meaningful for your target
WEATHER_MEAN_COLS = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]

FREQ = "30min"       # ORIGINAL: "15min"            # try 15min; if it's too big/slow, use "30min" or "1H"
TZ = "America/New_York"      # your local time zone
RUSH_AM = (6, 10)            # 7:00–10:00
RUSH_PM = (16, 19)           # 16:00–19:00

In [23]:
def make_aggregate_series(df_all: pd.DataFrame,
                          freq: str = FREQ,
                          station_col: str = "dock_name",
                          timestamp_col: str = "timestamp") -> pd.DataFrame:
    df = df_all.copy()
    df["ds"] = pd.to_datetime(df[timestamp_col], errors="coerce")
    df = df.dropna(subset=["ds"]).sort_values([station_col, "ds"])

    # If your timestamps are UTC, uncomment this and adjust as needed:
    # df["ds"] = df["ds"].dt.tz_localize("UTC").dt.tz_convert(TZ).dt.tz_localize(None)

    df["ds_bin"] = df["ds"].dt.floor(freq)

    # Build grid
    grid = pd.date_range(df["ds_bin"].min(), df["ds_bin"].max(), freq=freq)
    n = len(grid)

    # Sum “state at time” across stations using last-known value per bin and ffill
    y_total = np.zeros(n, dtype=float)
    state_totals = {c: np.zeros(n, dtype=float) for c in STATE_SUM_COLS}

    for _, g in df.groupby(station_col, sort=False):
        # last state in each bin for this station
        s = g.groupby("ds_bin", as_index=True).agg(
            occupancy=("occupancy", "last"),
            **{c: (c, "last") for c in STATE_SUM_COLS if c in g.columns}
        )
        s = s.reindex(grid).ffill()

        y_total += pd.to_numeric(s["occupancy"], errors="coerce").fillna(0.0).to_numpy()
        for c in state_totals:
            if c in s.columns:
                state_totals[c] += pd.to_numeric(s[c], errors="coerce").fillna(0.0).to_numpy()

    # Weather: compute mean per bin (usually shared signal)
    weather = (df.groupby("ds_bin")[ [c for c in WEATHER_MEAN_COLS if c in df.columns] ]
                 .mean()
                 .reindex(grid)
                 .interpolate(limit_direction="both"))

    agg = pd.DataFrame({"ds": grid, "y": y_total})
    for c, arr in state_totals.items():
        agg[c] = arr
    for c in weather.columns:
        agg[c] = weather[c].values

    # Holiday/weekend flags (if you already have them as columns, you can recompute here safely)
    agg["is_weekend"] = (agg["ds"].dt.weekday >= 5).astype(int)

    # Rush flags (weekday only)
    dow = agg["ds"].dt.weekday
    hour = agg["ds"].dt.hour + agg["ds"].dt.minute / 60.0
    is_weekday = dow < 5
    agg["is_rush_am"] = (is_weekday & (hour >= RUSH_AM[0]) & (hour < RUSH_AM[1])).astype(int)
    agg["is_rush_pm"] = (is_weekday & (hour >= RUSH_PM[0]) & (hour < RUSH_PM[1])).astype(int)

    agg['y_rate'] = agg['y'] / agg['tot_docks'].replace(0, 1)

    # Determine shift steps based on FREQ="30min"
    # 24 hours * 2 per hour = 48
    # 7 days * 48 per day = 336
    agg['y_lag_24h'] = agg['y_rate'].shift(48)
    agg['y_lag_7d'] = agg['y_rate'].shift(336)
    agg = agg.dropna(subset=['y_lag_24h', 'y_lag_7d'])

    return agg

In [25]:
CONDITION_COLS = ["is_rush_am", "is_rush_pm"]

def prep_dock_df(df_dock: pd.DataFrame):
    df = df_dock.copy()
    df["ds"] = pd.to_datetime(df["timestamp"])
    df["y"] = df["occupancy"].astype(float)

    # Ensure regressors are numeric
    for r in REGRESSORS + CONDITION_COLS:# ORIGINAL : REGRESSORS:
        if r in df.columns:
            df[r] = pd.to_numeric(df[r], errors="coerce")

    # Keep only needed columns
    cols = ["ds", "y"] + [r for r in REGRESSORS if r in df.columns]
    df = df[cols].dropna(subset=["ds", "y"]).sort_values("ds")

    return df

def prep_df(df_in: pd.DataFrame):
    df = df_in.copy()

    if "ds" not in df.columns:
        df["ds"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Normalize Target: y_rate = y (count) / tot_docks
    # Use 'occupancy' for the count if 'y' isn't already set to it, or strictly follow instruction to use 'occupancy'
    # The instruction says: Calculate the normalized target `y` as `occupancy` divided by `tot_docks`.
    occupancy_col = "occupancy" if "occupancy" in df.columns else "y"

    # Ensure numeric types
    occ = pd.to_numeric(df[occupancy_col], errors="coerce").fillna(0)
    docks = pd.to_numeric(df["tot_docks"], errors="coerce").fillna(1) # fill with 1 to avoid div/0

    df["y"] = occ / docks
    df["tot_docks"] = docks

    # Numeric cast regressors + conditions
    # Assuming CONDITION_COLS is defined globally from previous cells
    cols_to_numeric = REGRESSORS + (CONDITION_COLS if "CONDITION_COLS" in globals() else [])

    for c in cols_to_numeric:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    keep = ["ds", "y", "tot_docks"] + [c for c in cols_to_numeric if c in df.columns]
    df = df[keep].dropna(subset=["ds", "y"]).sort_values("ds")
    return df

In [27]:
def build_model(params):
    m = Prophet(
        daily_seasonality=False,   # we’ll define our own daily components
        weekly_seasonality=True,
        yearly_seasonality=False, #ORIGINAL : True,
        # ORIGINAL: default n_changepoints=25
        n_changepoints=10,
        **params
    )

    # Base within-day pattern
    m.add_seasonality(name="daily_base", period=1, fourier_order=5) # ORIGINAL: fourier_order=10)

    # Extra weekday rush-hour patterns
    m.add_seasonality(name="rush_am", period=1, fourier_order=4, condition_name="is_rush_am") #ORIGINAL : fourier_order=8
    m.add_seasonality(name="rush_pm", period=1, fourier_order=4, condition_name="is_rush_pm") #ORIGINAL : fourier_order=8

    # Add regressors (only if you truly want them)
    for r in REGRESSORS:
        m.add_regressor(r)  # must be present in both fit & predict dfs :contentReference[oaicite:4]{index=4}

    return m

In [29]:
def tune_prophet_for_dock(df_dock: pd.DataFrame,
                          horizon="7 days",
                          period="60 days",
                          initial="30 days"):
    df = prep_df(df_dock)

    param_grid = {
        "changepoint_prior_scale": [0.05, 0.1],
        "seasonality_prior_scale": [1.0, 10.0],
        "seasonality_mode": ["additive", "multiplicative"],
    }

    all_params = [
        dict(zip(param_grid.keys(), v))
        for v in itertools.product(*param_grid.values())
    ]

    results = []

    for params in all_params:
        m = build_model(params)
        print(df.columns)
        m.fit(df)

        df_cv = cross_validation(
            m,
            horizon=horizon,
            period=period,
            initial=initial,
            parallel=None # ORIGINAl : parallel="processes"
        )

        df_p = performance_metrics(df_cv, rolling_window=1)

        rmse = float(df_p["rmse"].mean())
        mae = float(df_p["mae"].mean())
        mape = float(df_p["mape"].mean()) if "mape" in df_p else np.nan

        results.append({
            **params,
            "rmse": rmse,
            "mae": mae,
            "mape": mape,
        })

    results_df = pd.DataFrame(results).sort_values("rmse")
    best = results_df.iloc[0][list(param_grid.keys())].to_dict()

    return best, results_df

In [31]:
df_all = pd.read_csv("data/CitiBikeDataMyBusProcessed_occupancyfix.csv")

REGRESSORS = [
    "is_weekend",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "y_lag_24h",
    "y_lag_7d"
]

agg = make_aggregate_series(df_all, freq=FREQ)

split_date = pd.Timestamp("2017-05-01")
train = agg[agg["ds"] < split_date].copy()
test  = agg[agg["ds"] >= split_date].copy()

best_params, leaderboard = tune_prophet_for_dock(
    train,                 # NOTE: now this is already the aggregate series
    horizon="30 days",
    period="60 days", # ORIGINAL : period="14 days",
    initial="120 days" # ORIGINAL :"180 days" . Made this change as
    #Minimum required: 180 + 30 + 28 = 238 days. That’s technically OK : but Prophet also drops the last cutoff, so borderline cases fail.
)
"""we change 180 to 120 as we still give Prophet 4 months of history
You allow multiple valid cutoffs
Runtime improves slightly"""

print("Best params:", best_params)
print(leaderboard.head(10))

Index(['ds', 'y', 'tot_docks', 'is_weekend', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'y_lag_24h',
       'y_lag_7d', 'is_rush_am', 'is_rush_pm'],
      dtype='object')


17:59:28 - cmdstanpy - INFO - Chain [1] start processing
17:59:33 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/11 [00:00<?, ?it/s]

17:59:33 - cmdstanpy - INFO - Chain [1] start processing
17:59:34 - cmdstanpy - INFO - Chain [1] done processing
17:59:35 - cmdstanpy - INFO - Chain [1] start processing
17:59:36 - cmdstanpy - INFO - Chain [1] done processing
17:59:37 - cmdstanpy - INFO - Chain [1] start processing
17:59:39 - cmdstanpy - INFO - Chain [1] done processing
17:59:39 - cmdstanpy - INFO - Chain [1] start processing
17:59:41 - cmdstanpy - INFO - Chain [1] done processing
17:59:42 - cmdstanpy - INFO - Chain [1] start processing
17:59:45 - cmdstanpy - INFO - Chain [1] done processing
17:59:45 - cmdstanpy - INFO - Chain [1] start processing
17:59:48 - cmdstanpy - INFO - Chain [1] done processing
17:59:49 - cmdstanpy - INFO - Chain [1] start processing
17:59:51 - cmdstanpy - INFO - Chain [1] done processing
17:59:52 - cmdstanpy - INFO - Chain [1] start processing
17:59:57 - cmdstanpy - INFO - Chain [1] done processing
17:59:58 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1]

Index(['ds', 'y', 'tot_docks', 'is_weekend', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'y_lag_24h',
       'y_lag_7d', 'is_rush_am', 'is_rush_pm'],
      dtype='object')


...

Index(['ds', 'y', 'tot_docks', 'is_weekend', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'y_lag_24h',
       'y_lag_7d', 'is_rush_am', 'is_rush_pm'],
      dtype='object')


18:04:42 - cmdstanpy - INFO - Chain [1] start processing
18:04:46 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/11 [00:00<?, ?it/s]

18:04:46 - cmdstanpy - INFO - Chain [1] start processing
18:04:48 - cmdstanpy - INFO - Chain [1] done processing
18:04:48 - cmdstanpy - INFO - Chain [1] start processing
18:04:49 - cmdstanpy - INFO - Chain [1] done processing
18:04:50 - cmdstanpy - INFO - Chain [1] start processing
18:04:51 - cmdstanpy - INFO - Chain [1] done processing
18:04:52 - cmdstanpy - INFO - Chain [1] start processing
18:04:54 - cmdstanpy - INFO - Chain [1] done processing
18:04:54 - cmdstanpy - INFO - Chain [1] start processing
18:04:56 - cmdstanpy - INFO - Chain [1] done processing
18:04:57 - cmdstanpy - INFO - Chain [1] start processing
18:04:58 - cmdstanpy - INFO - Chain [1] done processing
18:04:59 - cmdstanpy - INFO - Chain [1] start processing
18:05:02 - cmdstanpy - INFO - Chain [1] done processing
18:05:03 - cmdstanpy - INFO - Chain [1] start processing
18:05:07 - cmdstanpy - INFO - Chain [1] done processing
18:05:07 - cmdstanpy - INFO - Chain [1] start processing
18:05:11 - cmdstanpy - INFO - Chain [1]

Best params: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 1.0, 'seasonality_mode': 'multiplicative'}
   changepoint_prior_scale  seasonality_prior_scale seasonality_mode  \
1                     0.05                      1.0   multiplicative   
3                     0.05                     10.0   multiplicative   
5                     0.10                      1.0   multiplicative   
7                     0.10                     10.0   multiplicative   
2                     0.05                     10.0         additive   
0                     0.05                      1.0         additive   
6                     0.10                     10.0         additive   
4                     0.10                      1.0         additive   

       rmse       mae      mape  
1  0.000913  0.000537  0.085479  
3  0.000914  0.000537  0.085476  
5  0.000922  0.000548  0.086445  
7  0.000922  0.000548  0.086511  
2  0.000935  0.000555  0.086875  
0  0.000936  0.000556  0.08692

In [32]:

m = build_model(best_params)
m.fit(prep_df(train))

forecast_test = m.predict(prep_df(test).drop(columns=["y"]))
mae = mean_absolute_error(prep_df(test)["y"], forecast_test["yhat"])
print("Holdout MAE:", mae)

18:05:21 - cmdstanpy - INFO - Chain [1] start processing
18:05:25 - cmdstanpy - INFO - Chain [1] done processing


Holdout MAE: 0.0004767167441085446


In [33]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# Prepare train and test
df_train = prep_df(train)
df_test = prep_df(test)

# Forecast on test set
forecast_test = m.predict(df_test.drop(columns=["y"]))

# 1️⃣ Average occupancy
avg_train = df_train["y"].mean()
avg_test = df_test["y"].mean()

print(f"Average total occupancy (train): {avg_train:.2f} bikes")
print(f"Average total occupancy (test) : {avg_test:.2f} bikes")

# 2️⃣ MAE & relative error
mae = mean_absolute_error(df_test["y"], forecast_test["yhat"])
relative_error = mae / avg_test * 100
print(f"Holdout MAE: {mae:.2f} bikes")
print(f"Relative MAE (% of avg occupancy): {relative_error:.2f}%")

# 3️⃣ Print input features + actual y + predicted yhat
# Merge features with forecast for clarity
output_df = df_test.copy()
output_df["yhat"] = forecast_test["yhat"].values

# Optional: print first 10 rows for inspection
print("\nSample of features + actual + predicted:")
output_df.head(10)


Average total occupancy (train): 0.01 bikes
Average total occupancy (test) : 0.01 bikes
Holdout MAE: 0.00 bikes
Relative MAE (% of avg occupancy): 4.35%

Sample of features + actual + predicted:


,ds,y,tot_docks,is_weekend,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,y_lag_24h,y_lag_7d,is_rush_am,is_rush_pm,yhat
38016,2017-05-01 00:00:00,0.011605,22759.0,0,13.234083,85.879151,0.0,2.1,0.011605,0.013775,0,0,0.011634
38017,2017-05-01 00:30:00,0.011605,22759.0,0,13.219608,85.794514,0.0,2.1,0.011605,0.013862,0,0,0.011630
38018,2017-05-01 01:00:00,0.011605,22759.0,0,13.205133,85.709876,0.0,2.1,0.011605,0.013862,0,0,0.011632
38019,2017-05-01 01:30:00,0.011605,22759.0,0,13.190658,85.625239,0.0,2.1,0.011605,0.013862,0,0,0.011639
38020,2017-05-01 02:00:00,0.011605,22759.0,0,13.176184,85.540602,0.0,2.1,0.011605,0.013922,0,0,0.011648
38021,2017-05-01 02:30:00,0.011605,22759.0,0,13.161709,85.455964,0.0,2.1,0.011605,0.013922,0,0,0.011656
38022,2017-05-01 03:00:00,0.011605,22759.0,0,13.147234,85.371327,0.0,2.1,0.011605,0.013909,0,0,0.011658
38023,2017-05-01 03:30:00,0.011605,22759.0,0,13.132759,85.286689,0.0,2.1,0.011605,0.013909,0,0,0.011653
38024,2017-05-01 04:00:00,0.011605,22759.0,0,13.118285,85.202052,0.0,2.1,0.011605,0.013964,0,0,0.011642
38025,2017-05-01 04:30:00,0.011605,22759.0,0,13.103810,85.117414,0.0,2.1,0.011605,0.013964,0,0,0.011633


In [34]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# Prepare train and test
df_train = prep_df(train)
df_test = prep_df(test)

# Forecast on test set
forecast_test = m.predict(df_test.drop(columns=["y"]))

# Merge features with predictions
output_df = df_test.copy()
output_df["yhat"] = forecast_test["yhat"].values

# Compute absolute error for each row
output_df["abs_error"] = (output_df["y"] - output_df["yhat"]).abs()

# 1️⃣ Average occupancy
avg_train = df_train["y"].mean()
avg_test = df_test["y"].mean()

print(f"Average total occupancy (train): {avg_train:.2f} bikes")
print(f"Average total occupancy (test) : {avg_test:.2f} bikes")

# 2️⃣ MAE & relative error
mae = mean_absolute_error(df_test["y"], forecast_test["yhat"])
relative_error = mae / avg_test * 100
print(f"Holdout MAE: {mae:.2f} bikes")
print(f"Relative MAE (% of avg occupancy): {relative_error:.2f}%\n")

# 3️⃣ Sample of features + actual + predicted + error
print("Sample of features + actual + predicted + absolute error:")
print(output_df.head(10))

# 4️⃣ Top 10 largest errors
top_errors = output_df.sort_values("abs_error", ascending=False).head(50)
print("\nTop 10 largest errors:")
top_errors[20:]


Average total occupancy (train): 0.01 bikes
Average total occupancy (test) : 0.01 bikes
Holdout MAE: 0.00 bikes
Relative MAE (% of avg occupancy): 4.35%

Sample of features + actual + predicted + absolute error:
                       ds         y  tot_docks  is_weekend  temperature_2m  \
38016 2017-05-01 00:00:00  0.011605    22759.0           0       13.234083   
38017 2017-05-01 00:30:00  0.011605    22759.0           0       13.219608   
38018 2017-05-01 01:00:00  0.011605    22759.0           0       13.205133   
38019 2017-05-01 01:30:00  0.011605    22759.0           0       13.190658   
38020 2017-05-01 02:00:00  0.011605    22759.0           0       13.176184   
38021 2017-05-01 02:30:00  0.011605    22759.0           0       13.161709   
38022 2017-05-01 03:00:00  0.011605    22759.0           0       13.147234   
38023 2017-05-01 03:30:00  0.011605    22759.0           0       13.132759   
38024 2017-05-01 04:00:00  0.011605    22759.0           0       13.118285   
38025 20

,ds,y,tot_docks,is_weekend,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,y_lag_24h,y_lag_7d,is_rush_am,is_rush_pm,yhat,abs_error
53667,2018-03-23 01:30:00,0.012681,24094.0,0,7.200000,38.296398,0.000000,1.584163,0.000887,0.011999,0,0,0.002258,0.010423
53671,2018-03-23 03:30:00,0.012651,24094.0,0,5.600000,42.711005,0.000000,2.252036,0.000887,0.012085,0,0,0.002244,0.010407
53611,2018-03-21 21:30:00,0.000917,24094.0,0,0.000000,92.299311,0.243505,4.613595,0.011897,0.011725,0,0,0.011323,0.010406
53614,2018-03-21 23:00:00,0.000887,24094.0,0,0.583710,91.682784,1.300000,3.684163,0.012007,0.011765,0,0,0.011294,0.010406
53610,2018-03-21 21:00:00,0.000917,24094.0,0,-0.008157,92.298812,0.239426,4.878550,0.011912,0.011620,0,0,0.011315,0.010398
53670,2018-03-23 03:00:00,0.012640,24094.0,0,5.600000,42.711005,0.000000,2.252036,0.000887,0.012047,0,0,0.002246,0.010394
53662,2018-03-22 23:00:00,0.012515,24094.0,0,8.900000,32.676153,0.000000,5.700000,0.000887,0.011793,0,0,0.002122,0.010392
53666,2018-03-23 01:00:00,0.012631,24094.0,0,7.200000,38.296398,0.000000,1.584163,0.000887,0.011999,0,0,0.002246,0.010385
53664,2018-03-23 00:00:00,0.012552,24094.0,0,7.800000,36.760007,0.000000,3.697738,0.000887,0.011903,0,0,0.002176,0.010376
53663,2018-03-22 23:30:00,0.012515,24094.0,0,8.350000,34.718080,0.000000,4.698869,0.000887,0.011860,0,0,0.002150,0.010365


In [35]:
from prophet.serialize import model_to_json

with open("prophet_bike_model.json", "w") as f:
    f.write(model_to_json(m))